# **Emotion Classification _ EEG Signal _ An easy projects_ Shayan**

## Document

This notebook implements a complete deep learning pipeline for emotion classification using EEG (Electroencephalogram) signals. The dataset, containing multiple numerical EEG features and corresponding emotion labels, is first imported and preprocessed through several steps including label encoding, standardization, and data reshaping to ensure compatibility with different neural architectures. The features are normalized to zero mean and unit variance, and the dataset is divided into training and testing subsets to allow fair performance evaluation.

Three neural network models are designed and compared: a one-dimensional Convolutional Neural Network (CNN), a fully connected Artificial Neural Network (ANN), and a Long Short-Term Memory (LSTM) model. The CNN captures local spatial dependencies within EEG features using convolutional and pooling layers, while the ANN learns global feature relationships through dense connections. The LSTM model focuses on temporal dynamics, learning long-term dependencies across sequential EEG patterns. Each model employs regularization techniques such as Batch Normalization and Dropout to prevent overfitting, and all are trained using the Adam optimizer with categorical cross-entropy loss.

After training, model performance is evaluated on the test set using accuracy as the primary metric. The notebook thereby provides a comparative framework to assess how different deep learning architectures interpret EEG signals for emotion recognition tasks, highlighting their strengths in handling spatial, global, and temporal feature representations.

## Initials

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd

In [3]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization,LSTM


In [4]:
df = pd.read_csv('/content/drive/MyDrive/Emotion_Classification_EEG_Data_.csv')

In [ ]:
df.head()

,# mean_0_a,mean_1_a,mean_2_a,mean_3_a,mean_4_a,mean_d_0_a,mean_d_1_a,mean_d_2_a,mean_d_3_a,mean_d_4_a,...,fft_741_b,fft_742_b,fft_743_b,fft_744_b,fft_745_b,fft_746_b,fft_747_b,fft_748_b,fft_749_b,label
0,4.62,30.3,-356.0,15.6,26.3,1.070,0.411,-15.70,2.06,3.15,...,23.5,20.3,20.3,23.5,-215.0,280.00,-162.00,-162.00,280.00,NEGATIVE
1,28.80,33.1,32.0,25.8,22.8,6.550,1.680,2.88,3.83,-4.82,...,-23.3,-21.8,-21.8,-23.3,182.0,2.57,-31.60,-31.60,2.57,NEUTRAL
2,8.90,29.4,-416.0,16.7,23.7,79.900,3.360,90.20,89.90,2.03,...,462.0,-233.0,-233.0,462.0,-267.0,281.00,-148.00,-148.00,281.00,POSITIVE
3,14.90,31.6,-143.0,19.8,24.3,-0.584,-0.284,8.82,2.30,-1.97,...,299.0,-243.0,-243.0,299.0,132.0,-12.40,9.53,9.53,-12.40,POSITIVE
4,28.30,31.3,45.2,27.3,24.5,34.800,-5.790,3.06,41.40,5.52,...,12.0,38.1,38.1,12.0,119.0,-17.60,23.90,23.90,-17.60,NEUTRAL


In [5]:
#a total of 2132 rows and 2549 columns shoule be present
df.shape

(2132, 2549)

##  Preproceesing

In [6]:
# 📌 Separate features & labels
x = df.iloc[:, :-1].values  # EEG features (all columns except last)
y = df.iloc[:, -1].values   # Target labels (emotion category)


In [7]:
# Encoding labels (Converting text labels to numbers)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

#Normalizing EEG feature values
scaler = StandardScaler()
x = scaler.fit_transform(x)


In [8]:
#Reshape data for CNN input (CNN expects 3D input: samples, timesteps, features)
x = np.expand_dims(x, axis=2)  # Shape: (2132, 2548, 1)


In [9]:
#Splitting into train and test
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

##  CNN

In [10]:
model = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(x.shape[1], 1)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(filters=128, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(len(np.unique(y)), activation='softmax')  # Output layer for classification
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])



In [12]:
# 📌 Train the model
history = model.fit(xtrain, ytrain, epochs=30, batch_size=32, validation_data=(xtest, ytest))


Epoch 1/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 13s 103ms/step - accuracy: 0.7381 - loss: 8.5135 - val_accuracy: 0.4052 - val_loss: 11.0072
Epoch 2/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.8837 - loss: 2.0169 - val_accuracy: 0.3864 - val_loss: 26.5797
Epoch 3/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9171 - loss: 1.3410 - val_accuracy: 0.3489 - val_loss: 40.7486
Epoch 4/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9426 - loss: 0.4198 - val_accuracy: 0.4075 - val_loss: 26.7548
Epoch 5/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9392 - loss: 0.5926 - val_accuracy: 0.7518 - val_loss: 4.6904
Epoch 6/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9407 - loss: 0.4757 - val_accuracy: 0.6347 - val_loss: 6.0784
Epoch 7/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.9162 - loss: 0.4021 - val_accuracy: 0.9180 - val_loss: 0.6038
Epoch 8/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - accuracy: 0.9531 - loss: 0.3668 - val_accuracy: 0.90

In [13]:
test_loss, test_acc = model.evaluate(xtest, ytest)
print(f"Test Accuracy: {test_acc:.4f}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9887 - loss: 0.0907 
Test Accuracy: 0.9859


TEST ACC = 98.59%

## ANN

In [14]:
model1 = Sequential([
    Dense(512, activation='relu', input_shape=(x.shape[1],)),  # 🔹 Input layer: 2548 neurons
    BatchNormalization(),
    Dropout(0.3),

    Dense(256, activation='relu'),  # Hidden layer 1
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),  # Hidden layer 2
    Dropout(0.3),

    Dense(len(np.unique(y)), activation='softmax')  # Output layer for classification
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model1.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [16]:
model1.fit(xtrain, ytrain, epochs=30, batch_size=32, validation_data=(xtest, ytest))


Epoch 1/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 8s 66ms/step - accuracy: 0.7893 - loss: 0.5668 - val_accuracy: 0.9063 - val_loss: 0.2893
Epoch 2/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8925 - loss: 0.3019 - val_accuracy: 0.9297 - val_loss: 0.2270
Epoch 3/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9202 - loss: 0.2224 - val_accuracy: 0.9461 - val_loss: 0.1338
Epoch 4/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9361 - loss: 0.1678 - val_accuracy: 0.9532 - val_loss: 0.1298
Epoch 5/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9440 - loss: 0.1673 - val_accuracy: 0.9251 - val_loss: 0.1621
Epoch 6/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9596 - loss: 0.1262 - val_accuracy: 0.9415 - val_loss: 0.1399
Epoch 7/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9548 - loss: 0.1362 - val_accuracy: 0.9696 - val_loss: 0.0938
Epoch 8/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9668 - loss: 0.0901 - val_accuracy: 0.9602 - val_loss

In [17]:
test_loss, test_acc = model1.evaluate(xtest, ytest)
print(f"Test Accuracy: {test_acc:.4f}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9789 - loss: 0.0947 
Test Accuracy: 0.9766


TEST ACC: 97.66%

##  LSTM

In [18]:
model2 = Sequential([
    LSTM(128, return_sequences=True, input_shape=(x.shape[1], 1)),  # First LSTM layer
    BatchNormalization(),
    Dropout(0.3),

    LSTM(64, return_sequences=False),  # Second LSTM layer
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(len(np.unique(y)), activation='softmax')  # Output layer for classification
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [19]:
model2.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [20]:
model2.fit(xtrain, ytrain, epochs=30, batch_size=32, validation_data=(xtest, ytest))

Epoch 1/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 15s 167ms/step - accuracy: 0.6211 - loss: 0.9045 - val_accuracy: 0.3841 - val_loss: 1.0300
Epoch 2/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 160ms/step - accuracy: 0.7745 - loss: 0.5425 - val_accuracy: 0.3349 - val_loss: 1.2006
Epoch 3/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 8s 155ms/step - accuracy: 0.8128 - loss: 0.4702 - val_accuracy: 0.3372 - val_loss: 1.2563
Epoch 4/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 10s 186ms/step - accuracy: 0.8277 - loss: 0.4310 - val_accuracy: 0.7166 - val_loss: 1.3729
Epoch 5/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 160ms/step - accuracy: 0.8237 - loss: 0.4449 - val_accuracy: 0.5269 - val_loss: 3.2261
Epoch 6/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 12s 218ms/step - accuracy: 0.8677 - loss: 0.3405 - val_accuracy: 0.6019 - val_loss: 0.9736
Epoch 7/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 18s 164ms/step - accuracy: 0.8411 - loss: 0.3679 - val_accuracy: 0.7518 - val_loss: 1.2298
Epoch 8/30
54/54 ━━━━━━━━━━━━━━━━━━━━ 9s 166ms/step - accuracy: 0.8569 - loss: 0.3638 - val_accuracy

In [21]:
test_loss, test_acc = model2.evaluate(xtest, ytest)
print(f"Test Accuracy: {test_acc:.4f}")

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.9044 - loss: 0.2195
Test Accuracy: 0.9157


TEST ACC: 91.57%

## Question: shayanrokhva1999@gmail.com